# 02 — Bronze: raw listings per city

In [0]:
dbutils.widgets.text("city", "sicily")
dbutils.widgets.text("snapshot", "2026-09-25")
dbutils.widgets.text("source_path", "italy/sicilia/sicily")

city = dbutils.widgets.get("city").strip().lower()
snapshot = dbutils.widgets.get("snapshot").strip()
source_path = dbutils.widgets.get("source_path").strip().strip("/")

In [0]:
import os
import urllib.request
from pyspark.sql import functions as F

CATALOG = "workspace"
TABLE = f"{CATALOG}.bronze.listings"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.raw")

url = f"https://data.insideairbnb.com/{source_path}/{snapshot}/data/listings.csv.gz"
folder = f"/Volumes/{CATALOG}/bronze/raw/{city}/{snapshot}"
raw_file = f"{folder}/listings.csv.gz"

In [0]:
os.makedirs(folder, exist_ok=True)
if not os.path.exists(raw_file):
    urllib.request.urlretrieve(url, raw_file)

In [0]:
df = (spark.read
      .option("header", True)
      .option("multiLine", True)
      .option("escape", '"')
      .option("inferSchema", False)
      .csv(raw_file))

df = (df
      .withColumn("city", F.lit(city))
      .withColumn("snapshot_date", F.to_date(F.lit(snapshot)))
      .withColumn("_source_file", F.lit(raw_file))
      .withColumn("_ingested_at", F.current_timestamp()))

In [0]:
writer = df.write.format("delta").option("mergeSchema", "true")
if spark.catalog.tableExists(TABLE):
    writer.mode("overwrite").option("replaceWhere", f"city = '{city}'").saveAsTable(TABLE)
else:
    writer.mode("overwrite").partitionBy("city").saveAsTable(TABLE)

In [0]:
display(spark.table(TABLE)
        .groupBy("city", "snapshot_date")
        .agg(F.count("*").alias("rows"), F.countDistinct("id").alias("distinct_id"))
        .orderBy("city"))